In [57]:
import importlib
import subprocess
import sys
from typing import Optional


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception as e:
            raise ModuleNotFoundError(
                f"Missing dependency '{import_name}'. Tried to install '{pkg}' via pip but it failed. "
                f"Please install it in this environment (e.g. `pip install {pkg}`) and rerun. Original error: {e}"
            )


_ensure("datasets")
_ensure("pandas")
_ensure("matplotlib")
_ensure("seaborn")

from datasets import load_dataset

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover

    def display(x):
        print(x)

DATASET_NAME = "allenai/Dolci-RL-Zero-Math-7B"
SAMPLE_N = 1000
SAMPLE_SEED = 42

# Load dataset (handles DatasetDict vs Dataset)
ds = load_dataset(DATASET_NAME)
if hasattr(ds, "keys"):
    splits = list(ds.keys())
    dsets = [ds[s] for s in splits]
    d_all = pd.concat([d.to_pandas() for d in dsets], ignore_index=True)
else:
    splits = ["(single)"]
    d_all = ds.to_pandas()

# Random subsample to SAMPLE_N rows for faster iteration
if len(d_all) > SAMPLE_N:
    d_all = d_all.sample(n=SAMPLE_N, random_state=SAMPLE_SEED).reset_index(drop=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Splits loaded: {splits}")
print(f"Total rows (after subsample): {len(d_all)}  (seed={SAMPLE_SEED})")
print(f"Columns: {list(d_all.columns)}")

display(d_all.head(10))


Dataset: allenai/Dolci-RL-Zero-Math-7B
Splits loaded: ['train']
Total rows (after subsample): 1000  (seed=42)
Columns: ['prompt', 'ground_truth', 'messages', 'dataset']


,prompt,ground_truth,messages,dataset
0,"Call a fraction $\frac{a}{b}$, not necessarily...",11,"[{'content': 'Call a fraction $\frac{a}{b}$, n...",math
1,Three distinct vertices of a cube are chosen a...,11,[{'content': 'Three distinct vertices of a cub...,math
2,Find the minimum value of\n\[(12 - x)(10 - x)(...,-484,[{'content': 'Find the minimum value of \[(12 ...,math
3,Let $\mathcal{S}$ be the set of real numbers t...,360,[{'content': 'Let $\mathcal{S}$ be the set of ...,math
4,"In $\vartriangle ABC$ points $D, E$, and $F$ l...",94,"[{'content': 'In $\vartriangle ABC$ points $D,...",math
5,"Bill bought 13 notebooks, 26 pens, and 19 mark...",88,"[{'content': 'Bill bought 13 notebooks, 26 pen...",math
6,"Harold, Tanya, and Ulysses paint a very long p...",757,"[{'content': 'Harold, Tanya, and Ulysses paint...",math
7,Let $C$ be the number of ways to arrange the l...,126,[{'content': 'Let $C$ be the number of ways to...,math
8,"For how many triples of positive integers $(a,...",48,[{'content': 'For how many triples of positive...,math
9,"A laser is placed at the point $(3,5)$. The la...",10,[{'content': 'A laser is placed at the point $...,math


In [58]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

N_KEEP = 30
KEEP_SEED = 123

AIME_REPO = "talzoomanzoo/aime26"

USER_SUFFIX = "\n\nPresent the answer in LaTex format: \\boxed{Your answer}"

if "d_all" not in globals():
    raise RuntimeError("Run the cell above first to populate `d_all` (1k-subsampled Dolci frame).")

required = {"dataset", "prompt", "ground_truth"}
missing = required - set(d_all.columns)
if missing:
    raise KeyError(
        f"Dolci frame missing expected columns {sorted(missing)}. Has: {list(d_all.columns)}"
    )

d30 = d_all.sample(n=N_KEEP, random_state=KEEP_SEED).reset_index(drop=True)

aime_raw = load_dataset(AIME_REPO)
aime_split = aime_raw["train"] if hasattr(aime_raw, "keys") and "train" in aime_raw else (
    aime_raw[list(aime_raw.keys())[0]] if hasattr(aime_raw, "keys") else aime_raw
)
aime_df = aime_split.to_pandas()

need_cols = {"data_source", "prompt", "answer"}
missing_aime = need_cols - set(aime_df.columns)
if missing_aime:
    raise KeyError(
        f"{AIME_REPO} missing expected columns {sorted(missing_aime)}. Has: {list(aime_df.columns)}"
    )

def _extract_system_msg(prompt) -> str:
    """Extract the system-role content from a chat prompt (list/ndarray of dicts)."""
    if prompt is None:
        return ""
    try:
        items = list(prompt)
    except TypeError:
        return ""
    for item in items:
        if not isinstance(item, dict):
            continue
        if item.get("role") == "system" and item.get("content"):
            return str(item["content"])
    for item in items:
        if isinstance(item, dict) and item.get("content"):
            return str(item["content"])
    return ""


SYSTEM_MSG = _extract_system_msg(aime_df.iloc[0]["prompt"])
if not SYSTEM_MSG:
    raise RuntimeError(
        f"Could not extract a non-empty system message from {AIME_REPO}. "
        f"First prompt was: {aime_df.iloc[0]['prompt']!r}"
    )
print(f"Lifted system message from {AIME_REPO} ({len(SYSTEM_MSG)} chars)")


def _wrap_prompt(text) -> list[dict]:
    user_content = str(text).rstrip() + USER_SUFFIX
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_content},
    ]


def _norm_answer(x) -> str | None:
    if x is None:
        return None
    s = str(x).strip()
    return s if s else None


dolci_rows = pd.DataFrame(
    {
        "data_source": d30["dataset"].astype(str),
        "prompt": [_wrap_prompt(p) for p in d30["prompt"].tolist()],
        "answer": [_norm_answer(a) for a in d30["ground_truth"].tolist()],
    }
)
dolci_rows["member"] = 1

aime_rows = aime_df[["data_source", "prompt", "answer"]].copy()
aime_rows["answer"] = [_norm_answer(a) for a in aime_rows["answer"].tolist()]
aime_rows["member"] = 0

combined = pd.concat([aime_rows, dolci_rows], ignore_index=True)

print(f"aime26 rows (member=0): {len(aime_rows)}")
print(f"dolci  rows (member=1): {len(dolci_rows)}")
print(f"combined rows: {len(combined)}")
print(f"combined columns: {list(combined.columns)}")

BASE_DIR = Path(".") if Path("filter_benchmark_olmoe.ipynb").exists() else Path("benchmarks")
OUT_DIR = BASE_DIR / "filtered_samples"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_parquet = OUT_DIR / "talzoomanzoo__aime26_plus_dolci_rl_zero_math_7b__with_member.parquet"
combined.to_parquet(out_parquet, index=False)
print(f"Wrote: {out_parquet}")

display(combined.head(3))
display(combined.tail(3))


Lifted system message from talzoomanzoo/aime26 (385 chars)
aime26 rows (member=0): 60
dolci  rows (member=1): 30
combined rows: 90
combined columns: ['data_source', 'prompt', 'answer', 'member']
Wrote: filtered_samples/talzoomanzoo__aime26_plus_dolci_rl_zero_math_7b__with_member.parquet


,data_source,prompt,answer,member
0,aime26,[{'content': ' When tackling complex reasoning...,277,0
1,aime26,[{'content': ' When tackling complex reasoning...,62,0
2,aime26,[{'content': ' When tackling complex reasoning...,79,0


,data_source,prompt,answer,member
87,math,"[{'role': 'system', 'content': ' When tackling...",1514,1
88,math,"[{'role': 'system', 'content': ' When tackling...",216 \text{ inches},1
89,math,"[{'role': 'system', 'content': ' When tackling...",126,1


In [59]:
import importlib
import os
import subprocess
import sys


def _ensure(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
from huggingface_hub import HfApi

try:
    from huggingface_hub import get_token as _hf_get_token  # huggingface_hub >= 0.19
except ImportError:
    from huggingface_hub import HfFolder  # legacy

    def _hf_get_token():
        return HfFolder.get_token()


HF_REPO_ID = "talzoomanzoo/olmoe_member"

if "out_parquet" not in globals():
    raise RuntimeError(
        "Run the merge cell above first; `out_parquet` is not defined."
    )

token = _hf_get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `huggingface-cli login` or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
url = api.upload_file(
    path_or_fileobj=str(out_parquet),
    path_in_repo=out_parquet.name,
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message=f"Add {out_parquet.name} (aime26 + Dolci-RL-Zero-Math-7B, with member column)",
)
print(f"Uploaded to: {url}")


talzoomanzoo__aime26_plus_dolci_rl_zero_math_7b__with_member.parquet: 100%|██████████| 16.5k/16.5k [00:00<00:00, 57.5kB/s]


Uploaded to: https://huggingface.co/datasets/talzoomanzoo/olmoe_member/commit/dcb7e04a662a50bf0da569bec381106f8cbe760c


In [60]:
import os
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

OLMOE_MEMBER_REPO = "talzoomanzoo/olmoe_member"
EURUS_RL_REPO = "talzoomanzoo/eurus_rl_train_test"
EXCLUDE_DATA_SOURCES = {"numina_amc_aime", "aime26"}

olmoe_raw = load_dataset(OLMOE_MEMBER_REPO)
olmoe_split_name = (
    "train"
    if isinstance(olmoe_raw, DatasetDict) and "train" in olmoe_raw
    else (list(olmoe_raw.keys())[0] if isinstance(olmoe_raw, DatasetDict) else None)
)
olmoe = olmoe_raw[olmoe_split_name] if olmoe_split_name else olmoe_raw

need_cols = {"data_source", "prompt", "answer", "member"}
missing = need_cols - set(olmoe.column_names)
if missing:
    raise KeyError(
        f"{OLMOE_MEMBER_REPO} split={olmoe_split_name!r} missing {sorted(missing)}. "
        f"Has: {olmoe.column_names}"
    )

olmoe_n_before = len(olmoe)
olmoe = olmoe.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES)
print(
    f"{OLMOE_MEMBER_REPO}: {olmoe_n_before} -> {len(olmoe)} rows "
    f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES)}"
)


def _olmoe_to_rl(ex):
    ans = ex.get("answer")
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {
            "ground_truth": str(ans) if ans is not None else None,
            "style": "rule",
        },
        "member": ex.get("member"),
    }


olmoe_converted = olmoe.map(_olmoe_to_rl, remove_columns=olmoe.column_names)

eurus_rl = load_dataset(EURUS_RL_REPO)
if not isinstance(eurus_rl, DatasetDict) or not {"train", "test"}.issubset(eurus_rl.keys()):
    raise RuntimeError(
        f"{EURUS_RL_REPO} expected to be a DatasetDict with both 'train' and 'test' splits. "
        f"Got: {type(eurus_rl).__name__} keys={list(eurus_rl.keys()) if isinstance(eurus_rl, DatasetDict) else 'n/a'}"
    )

target_cols = ["data_source", "prompt", "reward_model", "member"]


def _prepare_eurus_split(name: str, ds):
    if "data_source" not in ds.column_names:
        raise KeyError(
            f"{EURUS_RL_REPO}[{name!r}] missing 'data_source' column. Has: {ds.column_names}"
        )
    n0 = len(ds)
    ds = ds.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES)
    print(
        f"{EURUS_RL_REPO}[{name!r}]: {n0} -> {len(ds)} rows "
        f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES)}"
    )
    if "member" not in ds.column_names:
        ds = ds.add_column("member", [0] * len(ds))
    missing = [c for c in target_cols if c not in ds.column_names]
    if missing:
        raise KeyError(
            f"{EURUS_RL_REPO}[{name!r}] missing target columns {missing}. Has: {ds.column_names}"
        )
    drop = [c for c in ds.column_names if c not in target_cols]
    if drop:
        ds = ds.remove_columns(drop)
    return ds


eurus_train = _prepare_eurus_split("train", eurus_rl["train"])
eurus_test = _prepare_eurus_split("test", eurus_rl["test"])

TEST_KEEP_DATA_SOURCES = {"aime", "aime25"}
_test_n_before = len(eurus_test)
eurus_test = eurus_test.filter(lambda ex: ex.get("data_source") in TEST_KEEP_DATA_SOURCES)
print(
    f"{EURUS_RL_REPO}['test']: {_test_n_before} -> {len(eurus_test)} rows "
    f"after keeping only data_source in {sorted(TEST_KEEP_DATA_SOURCES)}"
)
if len(eurus_test) == 0:
    raise RuntimeError(
        f"Test split is empty after restricting to data_source in {sorted(TEST_KEEP_DATA_SOURCES)}."
    )

MIX_SEED = 7

train_combined = concatenate_datasets([olmoe_converted, eurus_train]).shuffle(seed=MIX_SEED)
test_combined = eurus_test.shuffle(seed=MIX_SEED)

print(f"olmoe_member rows (data_source=math, member=1): {len(olmoe_converted)}")
print(f"eurus_rl train rows (member=0): {len(eurus_train)}")
print(f"eurus_rl test  rows (member=0): {len(eurus_test)}")
print(f"train rows: {len(train_combined)} (shuffled, seed={MIX_SEED})")
print(f"test  rows: {len(test_combined)} (shuffled, seed={MIX_SEED})")
print(f"columns: {train_combined.column_names}")
print(f"train member counts: {pd.Series(train_combined['member']).value_counts().to_dict()}")
print(f"test  member counts: {pd.Series(test_combined['member']).value_counts().to_dict()}")

train_sources = pd.Series(train_combined["data_source"]).value_counts()
test_sources = pd.Series(test_combined["data_source"]).value_counts()
unexpected_in_test = set(test_sources.index) - TEST_KEEP_DATA_SOURCES
if unexpected_in_test:
    raise RuntimeError(
        f"test split contains unexpected data_source values: {sorted(unexpected_in_test)}"
    )
print(f"train data_source counts:\n{train_sources}")
print(f"test  data_source counts:\n{test_sources}")

BASE_DIR = Path(".") if Path("filter_benchmark_olmoe.ipynb").exists() else Path("benchmarks")
OUT_DIR = BASE_DIR / "filtered_samples" / "olmoe_rl_train_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_parquet_train = OUT_DIR / "train.parquet"
out_parquet_test = OUT_DIR / "test.parquet"
train_combined.to_pandas().to_parquet(out_parquet_train, index=False)
test_combined.to_pandas().to_parquet(out_parquet_test, index=False)
print(f"Wrote: {out_parquet_train}")
print(f"Wrote: {out_parquet_test}")

display(train_combined.to_pandas().head(3))
display(train_combined.to_pandas().tail(3))
display(test_combined.to_pandas().head(3))


Generating train split: 90 examples [00:00, 16785.42 examples/s]
Filter: 100%|██████████| 90/90 [00:00<00:00, 9555.92 examples/s]


talzoomanzoo/olmoe_member: 90 -> 30 rows after dropping data_source in ['aime26', 'numina_amc_aime']


Map: 100%|██████████| 30/30 [00:00<00:00, 2553.97 examples/s]


talzoomanzoo/eurus_rl_train_test['train']: 1000 -> 970 rows after dropping data_source in ['aime26', 'numina_amc_aime']
talzoomanzoo/eurus_rl_train_test['test']: 100 -> 100 rows after dropping data_source in ['aime26', 'numina_amc_aime']
talzoomanzoo/eurus_rl_train_test['test']: 100 -> 100 rows after keeping only data_source in ['aime', 'aime25']
olmoe_member rows (data_source=math, member=1): 30
eurus_rl train rows (member=0): 970
eurus_rl test  rows (member=0): 100
train rows: 1000 (shuffled, seed=7)
test  rows: 100 (shuffled, seed=7)
columns: ['data_source', 'prompt', 'member', 'reward_model']
train member counts: {0: 970, 1: 30}
test  member counts: {0: 100}
train data_source counts:
olympiads        685
cn_contest       155
aops_forum        79
math              30
amc_aime          26
inequalities      14
olympiads_ref      9
number_theory      1
aime25             1
Name: count, dtype: int64
test  data_source counts:
aime25    52
aime      48
Name: count, dtype: int64
Wrote: fil

,data_source,prompt,member,reward_model
0,cn_contest,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': 'y=x^{2}-4 x+3', 'style': 'ru..."
1,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '3\frac{1}{3}', 'style': 'rule'}"
2,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '1', 'style': 'rule'}"


,data_source,prompt,member,reward_model
997,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '24', 'style': 'rule'}"
998,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '88', 'style': 'rule'}"
999,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '66', 'style': 'rule'}"


,data_source,prompt,reward_model,member
0,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '70', 'style': 'rule'}",0
1,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '60', 'style': 'rule'}",0
2,aime,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '073', 'style': 'rule'}",0


In [61]:
import importlib
import os
import subprocess
import sys


def _ensure(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
_ensure("datasets")
from huggingface_hub import HfApi
from datasets import Dataset, DatasetDict

try:
    from huggingface_hub import get_token as _hf_get_token  # huggingface_hub >= 0.19
except ImportError:
    from huggingface_hub import HfFolder  # legacy

    def _hf_get_token():
        return HfFolder.get_token()


HF_REPO_ID_RL = "talzoomanzoo/olmoe_rl_train_test_set"

for _name in ("out_parquet_train", "out_parquet_test"):
    if _name not in globals():
        raise RuntimeError(
            f"Run the mix cell above first; `{_name}` is not defined."
        )

token = _hf_get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `huggingface-cli login` or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID_RL, repo_type="dataset", exist_ok=True)

# Workaround for huggingface_hub<=1.12.x bug (#3829, fixed in #3847): commits
# rejected with "Your push was rejected because an LFS pointer pointed to a
# file that does not exist." Two defenses, both safe to apply unconditionally:
#   (1) Clear the in-process Xet write-token cache so we don't reuse a token
#       bound to a previously-deleted backing repo.
#   (2) Force `is_xet_available()` to return False for this process so the
#       upload path negotiates only `["basic", "multipart"]` LFS transfers and
#       skips the Xet code path entirely. `_commit_api._upload_files` reads
#       `constants.HF_HUB_DISABLE_XET` via `is_xet_available()` at runtime,
#       so monkeypatching after import works (no kernel restart needed).
try:
    from huggingface_hub.utils._xet import (
        XET_CONNECTION_INFO_CACHE,
        reset_xet_connection_info_cache_for_repo,
    )
    reset_xet_connection_info_cache_for_repo("dataset", HF_REPO_ID_RL)
    XET_CONNECTION_INFO_CACHE.clear()
    print("Cleared Xet connection info cache.")
except Exception as _exc:
    print(f"(Xet cache reset skipped: {type(_exc).__name__}: {_exc})")

import huggingface_hub.constants as _hf_constants
import huggingface_hub.utils._runtime as _hf_runtime
_hf_constants.HF_HUB_DISABLE_XET = True
os.environ["HF_HUB_DISABLE_XET"] = "1"
print(f"HF_HUB_DISABLE_XET set; is_xet_available()={_hf_runtime.is_xet_available()}")

if "train_combined" in globals() and "test_combined" in globals():
    train_ds = train_combined
    test_ds = test_combined
else:
    train_ds = Dataset.from_parquet(str(out_parquet_train))
    test_ds = Dataset.from_parquet(str(out_parquet_test))

dsd = DatasetDict({"train": train_ds, "test": test_ds})
print(
    f"Pushing DatasetDict to {HF_REPO_ID_RL}: "
    f"train={len(dsd['train'])}, test={len(dsd['test'])}"
)

commit_info = dsd.push_to_hub(
    repo_id=HF_REPO_ID_RL,
    token=token,
    commit_message=(
        "Update train/test splits "
        "(train: olmoe_member data_source=math member=1 + eurus_rl_train_test[train] "
        "minus {numina_amc_aime, aime26}; "
        "test: eurus_rl_train_test[test] restricted to data_source in {aime, aime25})"
    ),
)
print(f"Pushed. commit_info={commit_info}")
print(f"Load with: load_dataset('{HF_REPO_ID_RL}')")


Cleared Xet connection info cache.
HF_HUB_DISABLE_XET set; is_xet_available()=False
Pushing DatasetDict to talzoomanzoo/olmoe_rl_train_test_set: train=1000, test=100


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


Pushed. commit_info=https://huggingface.co/datasets/talzoomanzoo/olmoe_rl_train_test_set/commit/ce7f09ba6889a0a66008db57725854a4518fb4b8
Load with: load_dataset('talzoomanzoo/olmoe_rl_train_test_set')


In [62]:
import os
from pathlib import Path

import pandas as pd
import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

LIMR_MEMBER_REPO = "talzoomanzoo/limr_member"
EURUS_RL_REPO_LIMR = "talzoomanzoo/eurus_rl_train_test"
EXCLUDE_DATA_SOURCES_LIMR = {"numina_amc_aime", "aime26"}

# NOTE: we deliberately bypass `load_dataset(LIMR_MEMBER_REPO)` here. The
# combination of `datasets==3.2.0` and `huggingface_hub==1.12.2` is
# incompatible: `datasets` calls `dataclasses.fields(...)` on objects that
# `huggingface_hub>=1.0` no longer registers as `@dataclass`, raising
# `TypeError: must be called with a dataclass type or instance` whenever the
# parquet builder is invoked on a non-cached repo. Loading the parquet
# file(s) directly via `hf_hub_download` + `Dataset.from_parquet` sidesteps
# that code path entirely.
_hf_token = (
    huggingface_hub.get_token()
    or os.getenv("HF_TOKEN")
    or os.getenv("HUGGINGFACE_TOKEN")
)
_limr_api = HfApi(token=_hf_token)
_limr_info = _limr_api.dataset_info(LIMR_MEMBER_REPO)
_limr_parquet_files = sorted(
    s.rfilename for s in (_limr_info.siblings or []) if s.rfilename.endswith(".parquet")
)
if not _limr_parquet_files:
    raise RuntimeError(
        f"No .parquet files found in {LIMR_MEMBER_REPO}. Siblings: "
        f"{[s.rfilename for s in (_limr_info.siblings or [])]}"
    )
print(f"{LIMR_MEMBER_REPO} parquet files: {_limr_parquet_files}")

_limr_parts = [
    Dataset.from_parquet(
        hf_hub_download(LIMR_MEMBER_REPO, fname, repo_type="dataset", token=_hf_token)
    )
    for fname in _limr_parquet_files
]
limr = _limr_parts[0] if len(_limr_parts) == 1 else concatenate_datasets(_limr_parts)
limr_split_name = "(direct parquet)"

need_cols_limr = {"data_source", "prompt", "answer", "member"}
missing_limr = need_cols_limr - set(limr.column_names)
if missing_limr:
    raise KeyError(
        f"{LIMR_MEMBER_REPO} split={limr_split_name!r} missing {sorted(missing_limr)}. "
        f"Has: {limr.column_names}"
    )

limr_n_before = len(limr)
limr = limr.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES_LIMR)
print(
    f"{LIMR_MEMBER_REPO}: {limr_n_before} -> {len(limr)} rows "
    f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES_LIMR)}"
)


def _limr_to_rl(ex):
    ans = ex.get("answer")
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {
            "ground_truth": str(ans) if ans is not None else None,
            "style": "rule",
        },
        "member": ex.get("member"),
    }


limr_converted = limr.map(_limr_to_rl, remove_columns=limr.column_names)

eurus_rl_limr = load_dataset(EURUS_RL_REPO_LIMR)
if not isinstance(eurus_rl_limr, DatasetDict) or not {"train", "test"}.issubset(eurus_rl_limr.keys()):
    raise RuntimeError(
        f"{EURUS_RL_REPO_LIMR} expected to be a DatasetDict with both 'train' and 'test' splits. "
        f"Got: {type(eurus_rl_limr).__name__} "
        f"keys={list(eurus_rl_limr.keys()) if isinstance(eurus_rl_limr, DatasetDict) else 'n/a'}"
    )

target_cols_limr = ["data_source", "prompt", "reward_model", "member"]


def _prepare_eurus_split_limr(name: str, ds):
    if "data_source" not in ds.column_names:
        raise KeyError(
            f"{EURUS_RL_REPO_LIMR}[{name!r}] missing 'data_source' column. Has: {ds.column_names}"
        )
    n0 = len(ds)
    ds = ds.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES_LIMR)
    print(
        f"{EURUS_RL_REPO_LIMR}[{name!r}]: {n0} -> {len(ds)} rows "
        f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES_LIMR)}"
    )
    if "member" not in ds.column_names:
        ds = ds.add_column("member", [0] * len(ds))
    missing = [c for c in target_cols_limr if c not in ds.column_names]
    if missing:
        raise KeyError(
            f"{EURUS_RL_REPO_LIMR}[{name!r}] missing target columns {missing}. Has: {ds.column_names}"
        )
    drop = [c for c in ds.column_names if c not in target_cols_limr]
    if drop:
        ds = ds.remove_columns(drop)
    return ds


eurus_train_limr = _prepare_eurus_split_limr("train", eurus_rl_limr["train"])
eurus_test_limr = _prepare_eurus_split_limr("test", eurus_rl_limr["test"])

TEST_KEEP_DATA_SOURCES_LIMR = {"aime", "aime25"}
_test_n_before_limr = len(eurus_test_limr)
eurus_test_limr = eurus_test_limr.filter(
    lambda ex: ex.get("data_source") in TEST_KEEP_DATA_SOURCES_LIMR
)
print(
    f"{EURUS_RL_REPO_LIMR}['test']: {_test_n_before_limr} -> {len(eurus_test_limr)} rows "
    f"after keeping only data_source in {sorted(TEST_KEEP_DATA_SOURCES_LIMR)}"
)
if len(eurus_test_limr) == 0:
    raise RuntimeError(
        f"Test split is empty after restricting to data_source in {sorted(TEST_KEEP_DATA_SOURCES_LIMR)}."
    )

MIX_SEED_LIMR = 7

train_combined_limr = concatenate_datasets([limr_converted, eurus_train_limr]).shuffle(seed=MIX_SEED_LIMR)
test_combined_limr = eurus_test_limr.shuffle(seed=MIX_SEED_LIMR)

print(f"limr_member rows (member=1 surviving filter): {len(limr_converted)}")
print(f"eurus_rl train rows (member=0): {len(eurus_train_limr)}")
print(f"eurus_rl test  rows (member=0): {len(eurus_test_limr)}")
print(f"train rows: {len(train_combined_limr)} (shuffled, seed={MIX_SEED_LIMR})")
print(f"test  rows: {len(test_combined_limr)} (shuffled, seed={MIX_SEED_LIMR})")
print(f"columns: {train_combined_limr.column_names}")
print(f"train member counts: {pd.Series(train_combined_limr['member']).value_counts().to_dict()}")
print(f"test  member counts: {pd.Series(test_combined_limr['member']).value_counts().to_dict()}")

train_sources_limr = pd.Series(train_combined_limr["data_source"]).value_counts()
test_sources_limr = pd.Series(test_combined_limr["data_source"]).value_counts()
unexpected_in_test_limr = set(test_sources_limr.index) - TEST_KEEP_DATA_SOURCES_LIMR
if unexpected_in_test_limr:
    raise RuntimeError(
        f"test split contains unexpected data_source values: {sorted(unexpected_in_test_limr)}"
    )
print(f"train data_source counts:\n{train_sources_limr}")
print(f"test  data_source counts:\n{test_sources_limr}")

BASE_DIR_LIMR = Path(".") if Path("filter_benchmark_olmoe.ipynb").exists() else Path("benchmarks")
OUT_DIR_LIMR = BASE_DIR_LIMR / "filtered_samples" / "limr_rl_train_test"
OUT_DIR_LIMR.mkdir(parents=True, exist_ok=True)
out_parquet_train_limr = OUT_DIR_LIMR / "train.parquet"
out_parquet_test_limr = OUT_DIR_LIMR / "test.parquet"
train_combined_limr.to_pandas().to_parquet(out_parquet_train_limr, index=False)
test_combined_limr.to_pandas().to_parquet(out_parquet_test_limr, index=False)
print(f"Wrote: {out_parquet_train_limr}")
print(f"Wrote: {out_parquet_test_limr}")

display(train_combined_limr.to_pandas().head(3))
display(train_combined_limr.to_pandas().tail(3))
display(test_combined_limr.to_pandas().head(3))

talzoomanzoo/limr_member parquet files: ['talzoomanzoo__aime26_plus_still_amc_aime__with_member.parquet']


Generating train split: 60 examples [00:00, 11656.78 examples/s]
Filter: 100%|██████████| 60/60 [00:00<00:00, 6812.07 examples/s]


talzoomanzoo/limr_member: 60 -> 30 rows after dropping data_source in ['aime26', 'numina_amc_aime']


Filter: 100%|██████████| 1000/1000 [00:00<00:00, 32454.86 examples/s]


talzoomanzoo/eurus_rl_train_test['train']: 1000 -> 970 rows after dropping data_source in ['aime26', 'numina_amc_aime']


...

Filter: 100%|██████████| 100/100 [00:00<00:00, 12618.24 examples/s]

talzoomanzoo/eurus_rl_train_test['test']: 100 -> 100 rows after keeping only data_source in ['aime', 'aime25']
limr_member rows (member=1 surviving filter): 30
eurus_rl train rows (member=0): 970
eurus_rl test  rows (member=0): 100
train rows: 1000 (shuffled, seed=7)
test  rows: 100 (shuffled, seed=7)
columns: ['data_source', 'prompt', 'member', 'reward_model']
train member counts: {0: 970, 1: 30}
test  member counts: {0: 100}
train data_source counts:
olympiads        685
cn_contest       155
aops_forum        79
limr              30
amc_aime          26
inequalities      14
olympiads_ref      9
number_theory      1
aime25             1
Name: count, dtype: int64
test  data_source counts:
aime25    52
aime      48
Name: count, dtype: int64
Wrote: filtered_samples/limr_rl_train_test/train.parquet
Wrote: filtered_samples/limr_rl_train_test/test.parquet


,data_source,prompt,member,reward_model
0,cn_contest,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': 'y=x^{2}-4 x+3', 'style': 'ru..."
1,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '3\frac{1}{3}', 'style': 'rule'}"
2,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '1', 'style': 'rule'}"


,data_source,prompt,member,reward_model
997,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '24', 'style': 'rule'}"
998,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '88', 'style': 'rule'}"
999,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '66', 'style': 'rule'}"


,data_source,prompt,reward_model,member
0,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '70', 'style': 'rule'}",0
1,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '60', 'style': 'rule'}",0
2,aime,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '073', 'style': 'rule'}",0


In [63]:
import importlib
import os
import subprocess
import sys


def _ensure(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
_ensure("datasets")
from huggingface_hub import HfApi
from datasets import Dataset, DatasetDict

try:
    from huggingface_hub import get_token as _hf_get_token  # huggingface_hub >= 0.19
except ImportError:
    from huggingface_hub import HfFolder  # legacy

    def _hf_get_token():
        return HfFolder.get_token()


HF_REPO_ID_RL = "talzoomanzoo/limr_rl_train_test_set"

for _name in ("out_parquet_train_limr", "out_parquet_test_limr"):
    if _name not in globals():
        raise RuntimeError(
            f"Run the limr mix cell above first; `{_name}` is not defined."
        )

token = _hf_get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `huggingface-cli login` or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID_RL, repo_type="dataset", exist_ok=True)

# Workaround for huggingface_hub<=1.12.x bug (#3829, fixed in #3847): commits
# rejected with "Your push was rejected because an LFS pointer pointed to a
# file that does not exist." Two defenses, both safe to apply unconditionally:
#   (1) Clear the in-process Xet write-token cache so we don't reuse a token
#       bound to a previously-deleted backing repo.
#   (2) Force `is_xet_available()` to return False for this process so the
#       upload path negotiates only `["basic", "multipart"]` LFS transfers and
#       skips the Xet code path entirely. `_commit_api._upload_files` reads
#       `constants.HF_HUB_DISABLE_XET` via `is_xet_available()` at runtime,
#       so monkeypatching after import works (no kernel restart needed).
try:
    from huggingface_hub.utils._xet import (
        XET_CONNECTION_INFO_CACHE,
        reset_xet_connection_info_cache_for_repo,
    )
    reset_xet_connection_info_cache_for_repo("dataset", HF_REPO_ID_RL)
    XET_CONNECTION_INFO_CACHE.clear()
    print("Cleared Xet connection info cache.")
except Exception as _exc:
    print(f"(Xet cache reset skipped: {type(_exc).__name__}: {_exc})")

import huggingface_hub.constants as _hf_constants
import huggingface_hub.utils._runtime as _hf_runtime
_hf_constants.HF_HUB_DISABLE_XET = True
os.environ["HF_HUB_DISABLE_XET"] = "1"
print(f"HF_HUB_DISABLE_XET set; is_xet_available()={_hf_runtime.is_xet_available()}")

if "train_combined_limr" in globals() and "test_combined_limr" in globals():
    train_ds = train_combined_limr
    test_ds = test_combined_limr
else:
    train_ds = Dataset.from_parquet(str(out_parquet_train_limr))
    test_ds = Dataset.from_parquet(str(out_parquet_test_limr))

dsd = DatasetDict({"train": train_ds, "test": test_ds})
print(
    f"Pushing DatasetDict to {HF_REPO_ID_RL}: "
    f"train={len(dsd['train'])}, test={len(dsd['test'])}"
)

commit_info = dsd.push_to_hub(
    repo_id=HF_REPO_ID_RL,
    token=token,
    commit_message=(
        "Update train/test splits "
        "(train: limr_member member=1 + eurus_rl_train_test[train] "
        "minus {numina_amc_aime, aime26}; "
        "test: eurus_rl_train_test[test] restricted to data_source in {aime, aime25})"
    ),
)
print(f"Pushed. commit_info={commit_info}")
print(f"Load with: load_dataset('{HF_REPO_ID_RL}')")


Cleared Xet connection info cache.
HF_HUB_DISABLE_XET set; is_xet_available()=False
Pushing DatasetDict to talzoomanzoo/limr_rl_train_test_set: train=1000, test=100


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]


Pushed. commit_info=https://huggingface.co/datasets/talzoomanzoo/limr_rl_train_test_set/commit/d471eca945c93f7e303d78d971a4a38045940f19
Load with: load_dataset('talzoomanzoo/limr_rl_train_test_set')
